Objetivo: -   Calcule variaciones mensuales de volumen y precio

* Convertir data de raw a formato parquet
* guardar la data en interim
* escoger un producto en especifico para analizar y comenzar a armar las metricas

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import pyarrow

* Obteniendo la ruta de los  archivos

In [ ]:
from pathlib import Path

BASE_DIR=Path().resolve()
DATA_PATH=BASE_DIR.parent / 'data/raw'

files = list(DATA_PATH.glob('*.xlsx'))
for f in files:
    print(f)

* funcion para leer un excel y pasarlo a polars

In [ ]:
def read_excel_polars(file_path: Path) -> pl.DataFrame:
    df_pd=pd.read_excel(file_path)
    df_pl=pl.from_pandas(df_pd)

    return df_pl

In [ ]:
dfs=[read_excel_polars(file) for file in files]
df_all=pl.concat(dfs, how='vertical_relaxed')

In [ ]:
output_path=BASE_DIR.parent / 'data/interim'
df_all.write_parquet(output_path / 'df_all.parquet')

* Analisis de data y seleccion de data
* esto con el fin de trabajar con un solo ejemplo
* un solo producto 

In [2]:
from pathlib import Path

BASE_DIR=Path().resolve()
output_path=BASE_DIR.parent / 'data/interim'
df_final=pl.scan_parquet(output_path / 'df_all.parquet')

In [23]:
df_final.collect_schema()

Schema([('DÍA', Int64),
        ('MES', Int64),
        ('AÑO', Int64),
        ('DUA', Int64),
        ('SERIE', Int64),
        ('ADUANA', String),
        ('TIPO DOCUMENTO', String),
        ('RUC IMPORTADOR', Int64),
        ('IMPORTADOR', String),
        ('DIRECCIÓN', String),
        ('TELÉFONO', String),
        ('FAX', String),
        ('DEPARTAMENTO DE IMPORTADOR', String),
        ('PROVINCIA', String),
        ('DISTRITO', String),
        ('VÍA DE TRANSPORTE', String),
        ('BANCO', String),
        ('PAÍS DE ORIGEN', String),
        ('PAÍS DE ADQUISICIÓN', String),
        ('PUERTO DE EMBARQUE', String),
        ('PARTIDA ARANCELARIA', Int64),
        ('DESCRIPCIÓN ARANCELARIA', String),
        ('MERCANCÍA', String),
        ('DESCRIPCIÓN DE MERCANCÍA 1', String),
        ('DESCRIPCIÓN DE MERCANCÍA 2', String),
        ('DESCRIPCIÓN DE MERCANCÍA 3', String),
        ('DESCRIPCIÓN DE MERCANCÍA 4', String),
        ('PRODUCTO', String),
        ('MARCA', String),
    

In [ ]:
(
    df_final.group_by(pl.col('PARTIDA ARANCELARIA'))
    .agg(pl.col('US$ FOB').sum().alias('TOTAL_VALOR_FOB'))
    .sort('TOTAL_VALOR_FOB', descending=True)
    .collect()
).limit(20).to_pandas()

In [5]:
df_2710=df_final.filter(pl.col('PARTIDA ARANCELARIA') == 2710200012).collect()

El producto elegido esta asociado a la siguiente partida arancelaria
* Partida arancelaria 2710200012
* Descripcion arancelaria: Diesel B5, Con Un Contenido De Azufre Menor O Igual A 50 Ppm
* Producto: Diesel

Lo que se va calcular son las siguientes metricas
* Precio unitario mensual
* Volumen mensual
* Valor mensual

1. Construyes esto (nivel 1):

| periodo | precio | volumen |

In [3]:
def build_hs_monthly_base(
    df: pl.DataFrame,
    hs_code: str,
    hs_col: str = "hs_code",
    unit_col: str = "unidad_medida",
    value_col: str = "valor",
    quantity_col: str = "cantidad",
    day_col: str = "DIA",
    month_col: str = "MES",
    year_col: str = "AÑO",
) -> pl.DataFrame:
    return (
        df
        .filter(pl.col(hs_col) == hs_code)
        .filter(pl.col(unit_col).is_not_null())
        .with_columns([
            pl.col(day_col).cast(pl.Int32),
            pl.col(month_col).cast(pl.Int32),
            pl.col(year_col).cast(pl.Int32),
        ])
        .with_columns(
            pl.date(
                pl.col(year_col),
                pl.col(month_col),
                pl.col(day_col),
            ).alias("fecha")
        )
        .with_columns(
            pl.col("fecha").dt.truncate("1mo").alias("periodo")
        )
        .group_by(["periodo", hs_col, unit_col])
        .agg([
            pl.col(value_col).sum().alias("valor_total"),
            pl.col(quantity_col).sum().alias("volumen_total"),
        ])
        .with_columns(
            pl.when(pl.col("volumen_total") > 0)
            .then(pl.col("valor_total") / pl.col("volumen_total"))
            .otherwise(None)
            .alias("precio")
        )
        .select([
            "periodo",
            pl.col(hs_col).alias("hs_code"),
            pl.col(unit_col).alias("unidad_medida"),
            pl.col("volumen_total").alias("volumen"),
            "precio",
        ])
        .sort(["periodo", "unidad_medida"])
    )

In [10]:
df_2710=df_2710.with_columns(
    pl.col("PARTIDA ARANCELARIA").cast(str))

In [13]:
df_2710

DÍA,MES,AÑO,DUA,SERIE,ADUANA,TIPO DOCUMENTO,RUC IMPORTADOR,IMPORTADOR,DIRECCIÓN,TELÉFONO,FAX,DEPARTAMENTO DE IMPORTADOR,PROVINCIA,DISTRITO,VÍA DE TRANSPORTE,BANCO,PAÍS DE ORIGEN,PAÍS DE ADQUISICIÓN,PUERTO DE EMBARQUE,PARTIDA ARANCELARIA,DESCRIPCIÓN ARANCELARIA,MERCANCÍA,DESCRIPCIÓN DE MERCANCÍA 1,DESCRIPCIÓN DE MERCANCÍA 2,DESCRIPCIÓN DE MERCANCÍA 3,DESCRIPCIÓN DE MERCANCÍA 4,PRODUCTO,MARCA,MODELO,CARACTERÍSTICAS,AÑO FABRICACIÓN,US$ FOB,US$ FLETE,US$ SEGURO,US$ CIF,ADVALOREM,IGV,IPM,PESO NETO,PESO BRUTO,CANTIDAD,UNIDAD DE MEDIDA,US$ CIF UNIT,CANTIDAD COMERCIAL,UNIDAD COMERCIAL,TIPO DE BULTO,CANTIDAD BULTO,ESTADO DE MERCANCIA,PROBABLE EMBARCADOR,AGENTE DE ADUANA,EMPRESA DE TRANSPORTE,ALMACEN,INCOTERM,MANIFIESTO DE CARGA,FECHA DE LLEGADA,US$ FOB UNIT,BILL OF LADING MASTER,BILL OF LADING,FECHA DE EMBARQUE,BUQUE,AGENTE DE CARGA EN DESTINO,FORMA DE PAGO,CANAL
i64,i64,i64,i64,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,str,f64,str,str,str,str,str,str,i64,f64,f64,str,str,f64,str,str,str,str
23,1,2025,88,1,"""MOLLENDO - MATARANI""","""RUC - Reg. Unico Contribuyente""",20603111304,"""MOBIL PETROLEUM OVERSEAS COMPA…",null,null,null,null,null,null,"""MARITIMO""","""BANCO CITIBANK NA""","""ESTADOS UNIDOS""","""ESTADOS UNIDOS""","""BATON ROUGE, LA, ESTADOS UNIDO…","""2710200012.0""","""Diesel B5, Con Un Contenido De…","""DIESEL B5 S-50, S/M, S/M""",null,"""40,000.000 BARRILES DE DIESEL …","""GRANEL""","""DIESEL B5 S-50, CONTRATO:9181…",null,null,null,null,null,6.2340e6,597414.0,1301.17,6.8328e6,0.0,1.0932e6,136655.28,8.6698e6,8.6698e6,10343.51,"""M3""",660.59,65058.7,"""BARRIL""","""GRANEL""",8.6698e6,"""NUEVO""","""N-A""","""INTERAMERICAN SERVICE CO S.A.C…",null,"""MONTE AZUL SUR SOCIEDAD ANONIM…",null,491,2.0250112e7,602.701489,null,null,null,null,"""IAN TAYLOR PERU S.A.C""",null,"""VERDE"""
23,1,2025,23,1,"""SALAVERRY""","""RUC - Reg. Unico Contribuyente""",20554545743,"""CORPORACION PRIMAX S.A.""",null,null,null,null,null,null,"""MARITIMO""","""BANCO INTERNACIONAL DEL PERU""","""ESTADOS UNIDOS""","""ESTADOS UNIDOS""","""HOUSTON, TX, ESTADOS UNIDOS; U…","""2710200012.0""","""Diesel B5, Con Un Contenido De…","""DIESEL B5, S/M, S/M""","""CON AZUFRE MENOS DE 10 PPM""","""COMBUSTIBLE""","""DIESEL B5 WITH SULFURLESS THAN…","""CORRESPONDE A 29,980.64 BARRI…","""DIESEL B5""","""S/M""","""S/M""","""DIESEL B5 WITH SULFURLESS THAN…",null,2.9742e6,312656.0,1234.0,3.2881e6,0.0,521256.96,65157.12,4.009341e6,4.009341e6,4766.54,"""M3""",689.84,1.2592e6,"""GALON""","""KEG""",4.009341e6,"""NUEVO""","""N-A""","""SCHARFF LOGISTICA INTEGRADA S.…","""COSMOS AGENCIA MARITIMA SAC""","""CORPORACION PRIMAX S.A.""",null,289,2.0250125e7,623.985067,null,null,null,null,null,null,"""NARANJA"""
3,1,2025,3302,1,"""MARITIMA DEL CALLAO""","""RUC - Reg. Unico Contribuyente""",20513251506,"""VALERO PERU S.A.C.""","""AV. CANAVAL Y MOREYRA NRO. 380…","""6169292""","""'-""","""LIMA""","""LIMA""","""SAN ISIDRO""","""MARITIMO""",null,"""ESTADOS UNIDOS""","""ESTADOS UNIDOS""","""HOUSTON, TX, ESTADOS UNIDOS; U…","""2710200012.0""","""Diesel B5, Con Un Contenido De…","""DIESEL B5, S/M, S/M""","""A GRANEL""","""29,690.54 BARRILES = 1,247,002…","""DIESEL DB5 WITH SULFUR LESS TH…","""CONTRATO N? 8275591 FECHA: 19/…","""DIESEL B5""","""S/M""","""S/M""","""DIESEL DB5 WITH SULFUR LESS TH…",null,2702424.4,241574.0,1236.9,2945235.3,0.0,476486.44,59560.81,4.004267e6,4.004267e6,4720.42,"""M3""",623.93,1.2470e6,"""GALON""","""KILOGRAMOS""",4.004267e6,"""NUEVO""","""BP PRODUCTS NORTH AMERICA INC""","""ADUAMERICA S.A.""","""COSMOS AGENCIA MARITIMA SAC""","""VALERO PERU S.A.C.""","""CIF""",3426,2.0250104e7,572.4966,"""2825519-2""","""2825519-2""",2.0241217e7,"""QUARTZ""",null,"""PAGO DIFERIDO""","""VERDE"""
3,1,2025,4,1,"""MOLLENDO - MATARANI""","""RUC - Reg. Unico Contribuyente""",20603111304,"""MOBIL PETROLEUM OVERSEAS COMPA…",null,null,null,null,null,null,"""MARITIMO""

In [12]:
build_hs_monthly_base(
    df_2710,
    hs_code = "2710200012",
    hs_col = "PARTIDA ARANCELARIA",
    unit_col = "UNIDAD DE MEDIDA",
    value_col = "US$ FOB",
    quantity_col = "CANTIDAD",
    day_col = "DÍA",
    month_col = "MES",
    year_col = "AÑO",
)

periodo,hs_code,unidad_medida,volumen,precio
date,str,str,f64,f64
